In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import harmonypy as hm
import scanorama

script_loc = Path('01_load_and_preprocess.py').resolve().parent
data_loc = script_loc.parent / "data"
resources_loc = script_loc.parent / 'resources'

In [ ]:
#Read in preprocessed data
cdata = sc.read_h5ad(data_loc / 'intermediate/c_preintegration.h5ad')
pdata = sc.read_h5ad(data_loc / 'intermediate/p_preintegration.h5ad')

In [ ]:
#Copy objects to enable comparison of integration methods
hcdata = cdata.copy()
scdata = cdata.copy()

hpdata = pdata.copy()
spdata = pdata.copy()

In [ ]:
#Integrate data using harmonypy
pcs = hcdata.obsm['X_pca']
harmony_out = hm.run_harmony(pcs, hcdata.obs, "id")
hcdata.obsm['X_pca_harmony'] = harmony_out.Z_corr

pcs = hpdata.obsm['X_pca']
harmony_out = hm.run_harmony(pcs, hpdata.obs, "id")
hpdata.obsm['X_pca_harmony'] = harmony_out.Z_corr

In [ ]:
#Calculate neighbors and umap on corrected pcs
sc.pp.neighbors(hcdata, use_rep='X_pca_harmony')
sc.tl.umap(hcdata)

sc.pp.neighbors(hpdata, use_rep='X_pca_harmony')
sc.tl.umap(hpdata)

In [ ]:
#Integrate data using scanorama

#Make list of batches
scdata['raw_counts'] = scdata.X.copy()
scdata.X = scdata.layers['CP10k_counts']
batch_key = 'id'
mdatas = [scdata[scdata.obs[batch_key] == batch_value].copy() for batch_value in scdata.obs[batch_key].unique()]

#Run scanorama and collect correct space
scanorama.integrate_scanpy(mdatas, dimred = 100)
scanorama_int = [ad.obsm['X_scanorama'] for ad in mdatas]

all_s = np.concatenate(scanorama_int)

scdata.obsm['X_scanorama'] = all_s

#Make list of batches
spdata['raw_counts'] = spdata.X.copy()
spdata.X = spdata.layers['CP10k_counts']
batch_key = 'id'
mdatas = [spdata[spdata.obs[batch_key] == batch_value].copy() for batch_value in spdata.obs[batch_key].unique()]

#Run scanorama and collect correct space
scanorama.integrate_scanpy(mdatas, dimred = 100)
scanorama_int = [ad.obsm['X_scanorama'] for ad in mdatas]

all_s = np.concatenate(scanorama_int)

spdata.obsm['X_scanorama'] = all_s

In [ ]:
#Calculate neighbors and umap on corrected pcs
sc.pp.neighbors(scdata, use_rep='X_scanorama')
sc.tl.umap(scdata)

sc.pp.neighbors(spdata, use_rep='X_scanorama')
sc.tl.umap(spdata)